# 🤖 Working with LLMs & Foundation Models

## 1. What is a Foundation Model?

A **foundation model** is a large AI model trained on vast amounts of data (text, code, images) that can be adapted to many different tasks.

### What makes them different?
Traditional AI models are usually trained for one narrow job, like classifying spam or recognizing faces. Foundation models are designed to be more general: after pretraining, they can be fine-tuned or prompted to do many tasks, such as writing text, answering questions, generating images, or analyzing code.

Think of it like this:

| Old approach | Foundation model approach |
|---|---|
| Train a model *for each task* | Train **one giant model**, adapt for many tasks |
| Needs labelled data per task | Can work with just a description (prompt) |
| Weeks of training per model | Reuse the same base — just engineer your prompt |

### The LLM Family Tree

```
Foundation Models
├── Text (LLMs)      → GPT-4, Claude, Gemini, Llama 3
├── Image            → DALL·E, Stable Diffusion, Midjourney  
├── Code             → GitHub Copilot, CodeLlama
└── Multimodal       → GPT-4o, Claude 3 (text + images + code)
```

### How does an LLM actually work?

An LLM is trained to **predict the next token** (word piece). That's it.

```
Input:  "The capital of France is"
Model:  [looks at all tokens, calculates attention weights]
Output: "Paris"  ← the most probable next token
```

Through training on trillions of tokens, the model internalises:
- Grammar and language structure
- World knowledge
- Reasoning patterns
- Code syntax


### Visual: How a prompt flows through an LLM

```
┌─────────────────────────────────────────────────────────────┐
│                        YOUR PROMPT                          │
│   "Summarise this customer review in one sentence."        │
└────────────────────────┬────────────────────────────────────┘
                         │
                         ▼
┌─────────────────────────────────────────────────────────────┐
│                     TOKENISATION                            │
│   ["Summar","ise"," this"," customer"," review"," in"...]  │
└────────────────────────┬────────────────────────────────────┘
                         │
                         ▼
┌─────────────────────────────────────────────────────────────┐
│               TRANSFORMER LAYERS (x96 in GPT-4)            │
│  Each layer: Attention → which tokens matter to each other  │
│              Feed-forward → transform the representation    │
└────────────────────────┬────────────────────────────────────┘
                         │
                         ▼
┌─────────────────────────────────────────────────────────────┐
│                    OUTPUT (tokens)                          │
│   "This review praises the fast delivery and packaging."   │
└─────────────────────────────────────────────────────────────┘
```

**Key insight:** You don't need to understand every transformer layer to *use* LLMs effectively — just like you don't need to understand TCP/IP to use the internet.


## 2. Calling an LLM API in Python

In practice, you'll interact with LLMs through an API. Let's see how.

### Installation
```bash
pip install anthropic openai  # install one or both
```


In [6]:
# We'll simulate LLM responses for this notebook so it runs without API keys.

import json, textwrap

def simulate_llm(prompt, system=None, temperature=0.7):
    """Simulates an LLM response for teaching purposes."""
    responses = {
        "sentiment": "POSITIVE — The reviewer is happy with delivery speed and packaging quality.",
        "extract": json.dumps({"product": "headphones", "rating": 4, "issue": "ear cushions wear out quickly", "would_recommend": True}),
        "haiku": "Golden autumn leaves\nFall gently on quiet streams\nNature breathes and rests",
        "classify": "Category: BILLING\nPriority: HIGH\nSuggested action: Escalate to billing team within 24 hours",
        "chain1": "The patient reports chest pain radiating to the left arm.",
        "chain2": "Symptoms (chest pain, left arm radiation) may indicate cardiac event. Recommend immediate ECG.",
        "chain3": "URGENT: Possible cardiac event. Action: Call emergency services immediately.",
        "rag": "Based on the provided documents:\n\nThe Q3 2024 return policy states that items can be returned within 30 days with receipt. Electronics have a 15-day return window. Source: [Policy Doc, Section 3.2]",
    }
    for key, val in responses.items():
        if key in prompt.lower() or (system and key in system.lower()):
            return val
    return "This is a simulated LLM response. In production, replace simulate_llm() with a real API call."

print("✅ Simulation ready. Here's what a real API call looks like:")
print("""
# REAL ANTHROPIC API (swap in when you have a key)
import anthropic
client = anthropic.Anthropic(api_key="your-key-here")

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024,
    system="You are a helpful data science assistant.",
    messages=[{"role": "user", "content": "Explain overfitting in simple terms."}]
)
print(response.content[0].text)
""")


✅ Simulation ready. Here's what a real API call looks like:

# REAL ANTHROPIC API (swap in when you have a key)
import anthropic
client = anthropic.Anthropic(api_key="your-key-here")

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024,
    system="You are a helpful data science assistant.",
    messages=[{"role": "user", "content": "Explain overfitting in simple terms."}]
)
print(response.content[0].text)



## 3. Prompt Engineering

**Prompt engineering** is the art of writing instructions to get reliable, useful outputs from an LLM.

### The anatomy of a good prompt

| Component | Purpose | Example |
|---|---|---|
| **System prompt** | Set the model's role/persona | "You are a data analyst who communicates clearly to non-technical stakeholders." |
| **Task instruction** | Tell it exactly what to do | "Classify the following customer complaint into one of: BILLING, SHIPPING, PRODUCT, OTHER." |
| **Context** | Provide relevant background | "Our company sells electronics. The complaint is: ..." |
| **Output format** | Specify the shape of the answer | "Respond with JSON only: {category, priority, action}" |
| **Examples** | Show, don't just tell (few-shot) | "Example — Input: 'My order never arrived' → Output: SHIPPING" |


In [4]:
# ── Prompt Engineering Examples ──────────────────────────────────

# Example 1: ZERO-SHOT (no examples, just instructions)
print("=" * 55)
print("EXAMPLE 1: Zero-shot sentiment analysis")
print("=" * 55)

zero_shot_prompt = """
Analyse the sentiment of this product review.
Respond with: POSITIVE, NEGATIVE, or NEUTRAL, 
then a one-sentence reason.

Review: "Arrived two days early and the packaging was immaculate. 
Very impressed!"
"""

response = simulate_llm("sentiment")
print(f"Prompt:\n{zero_shot_prompt.strip()}")
print(f"\nLLM Response:\n{response}")

EXAMPLE 1: Zero-shot sentiment analysis
Prompt:
Analyse the sentiment of this product review.
Respond with: POSITIVE, NEGATIVE, or NEUTRAL, 
then a one-sentence reason.

Review: "Arrived two days early and the packaging was immaculate. 
Very impressed!"

LLM Response:
POSITIVE — The reviewer is happy with delivery speed and packaging quality.


In [10]:
# Example 2: STRUCTURED OUTPUT (asking for JSON)
print("=" * 55)
print("EXAMPLE 2: Structured output (JSON extraction)")
print("=" * 55)

structured_prompt = """
Extract key information from this review as JSON.
Return ONLY valid JSON, no other text.

Schema:
{
  "product": string,
  "rating": int (1-5),
  "issue": string or null,
  "would_recommend": boolean
}

Review: "These headphones sound great (4/5) but the ear 
cushions wear out after 6 months. Would still recommend 
for the price."
"""

response = simulate_llm("extract")
print(f"\nLLM Response (JSON):\n{response}")

# Parse the JSON like you would in a real pipeline
import json
parsed = json.loads(response)
print(f"\nParsed — Product: {parsed['product']}, Rating: {parsed['rating']}/5")


EXAMPLE 2: Structured output (JSON extraction)

LLM Response (JSON):
{"product": "headphones", "rating": 4, "issue": "ear cushions wear out quickly", "would_recommend": true}

Parsed — Product: headphones, Rating: 4/5


In [11]:
# Example 3: FEW-SHOT (providing examples to guide the model)
print("=" * 55)
print("EXAMPLE 3: Few-shot customer ticket classification")
print("=" * 55)

few_shot_prompt = """
Classify customer support tickets. 

Examples:
Input: "I was charged twice for my order last week"
Output: Category: BILLING | Priority: HIGH | Action: Escalate to billing team

Input: "My package says delivered but I never received it"  
Output: Category: SHIPPING | Priority: MEDIUM | Action: Open courier investigation

Now classify this ticket:
Input: "I've been waiting 3 days and my refund hasn't appeared on my statement"
"""

response = simulate_llm("classify")
print(f"\nLLM Response:\n{response}")


EXAMPLE 3: Few-shot customer ticket classification

LLM Response:
Category: BILLING
Priority: HIGH
Suggested action: Escalate to billing team within 24 hours


In [12]:
# Example 4: CHAIN OF THOUGHT (asking the model to reason step by step)
print("=" * 55)
print("EXAMPLE 4: Chain-of-thought reasoning")
print("=" * 55)

# Simulate a 3-step chain
print("Step 1 — Extract symptoms:")
step1 = simulate_llm("chain1")
print(f"  → {step1}")

print("\nStep 2 — Medical assessment:")
step2 = simulate_llm("chain2")
print(f"  → {step2}")

print("\nStep 3 — Recommend action:")
step3 = simulate_llm("chain3")
print(f"  → {step3}")

print("""
💡 Key insight: Breaking a complex task into sequential prompts 
   (a "chain") often gives better results than one giant prompt.
   This is called a PIPELINE or CHAIN architecture.
""")


EXAMPLE 4: Chain-of-thought reasoning
Step 1 — Extract symptoms:
  → The patient reports chest pain radiating to the left arm.

Step 2 — Medical assessment:
  → Symptoms (chest pain, left arm radiation) may indicate cardiac event. Recommend immediate ECG.

Step 3 — Recommend action:
  → URGENT: Possible cardiac event. Action: Call emergency services immediately.

💡 Key insight: Breaking a complex task into sequential prompts 
   (a "chain") often gives better results than one giant prompt.
   This is called a PIPELINE or CHAIN architecture.



---
## 4. Retrieval-Augmented Generation (RAG)

**The problem:** LLMs have a knowledge cutoff and don't know *your* documents.

**The solution:** Before asking the LLM your question, *retrieve* relevant documents and include them in the prompt.

```
WITHOUT RAG:
  User: "What is our return policy?"
  LLM:  "I don't have access to your specific policies..." ❌

WITH RAG:
  1. Search your document store for "return policy"
  2. Find: Policy Doc Section 3.2 — "30 days with receipt..."
  3. Include that in the prompt
  LLM:  "Based on your policy document, items can be returned
          within 30 days with receipt. Electronics: 15 days." ✅
```

### RAG Architecture

```
                    ┌─────────────────────┐
  Your docs ──────▶ │   Vector Database   │
  (PDFs, wikis,     │  (semantic search)  │
   policies, etc.)  └──────────┬──────────┘
                               │ retrieve top-k chunks
                               ▼
  User question ─────▶ ┌──────────────┐
                        │   Augmented  │ = question + retrieved chunks
                        │    Prompt    │
                        └──────┬───────┘
                               │
                               ▼
                        ┌──────────────┐
                        │     LLM      │ ──▶ Grounded answer + source
                        └──────────────┘
```


In [5]:
# ── Simple RAG simulation ─────────────────────────────────────────────────

# Step 1: A tiny "document store" (in production: a vector database)
documents = [
    {"id": "policy_3_2", "text": "Items can be returned within 30 days with receipt. "
                                  "Electronics have a 15-day return window."},
    {"id": "policy_5_1", "text": "Refunds are processed within 5-7 business days "
                                  "back to the original payment method."},
    {"id": "shipping_1", "text": "Standard shipping takes 3-5 business days. "
                                  "Express shipping delivers next day for orders placed before 2pm."},
]

# Step 2: A simple keyword retrieval function
# (In production: embedding similarity search)
def retrieve(query, docs, top_k=2):
    query_words = set(query.lower().split())
    scored = []
    for i, doc in enumerate(docs):
        overlap = len(query_words & set(doc["text"].lower().split()))
        scored.append((overlap, i, doc))  # add index `i` as tiebreaker
    scored.sort(reverse=True)
    return [doc for _, _, doc in scored[:top_k]]

# Step 3: Build the augmented prompt
user_question = "What is your return policy for electronics?"
retrieved = retrieve(user_question, documents)

augmented_prompt = f"""
Answer the user's question using ONLY the provided documents.
If the answer isn't in the documents, say "I don't have that information."
Cite the source ID at the end.

Documents:
{chr(10).join(f'[{d["id"]}]: {d["text"]}' for d in retrieved)}

User question: {user_question}
"""

# Step 4: Send to LLM
response = simulate_llm("rag")

print(f"User question: {user_question}")
print(f"\nRetrieved {len(retrieved)} relevant chunks:")
for doc in retrieved:
    print(f"  [{doc['id']}]: {doc['text'][:60]}...")
print(f"\nLLM Answer (grounded):")
print(response)


User question: What is your return policy for electronics?

Retrieved 2 relevant chunks:
  [shipping_1]: Standard shipping takes 3-5 business days. Express shipping ...
  [policy_3_2]: Items can be returned within 30 days with receipt. Electroni...

LLM Answer (grounded):
Based on the provided documents:

The Q3 2024 return policy states that items can be returned within 30 days with receipt. Electronics have a 15-day return window. Source: [Policy Doc, Section 3.2]


## 5. When to Use an LLM API vs. Train Your Own Model

This is one of the most important judgment calls in modern data science.

| Situation | Use LLM API | Train your own |
|---|---|---|
| **Task type** | Language, reasoning, generation | Structured/tabular prediction |
| **Data volume** | Low (dozens of examples) | High (thousands+) |
| **Latency** | Can tolerate 1–5 seconds | Need <100ms |
| **Privacy** | Data can leave your servers | Sensitive data must stay on-premise |
| **Cost** | Low volume or prototyping | High volume in production |
| **Accuracy** | Good enough with prompting | Need fine-tuned precision |
| **Interpretability** | Not critical | Regulators require explanations |

### Decision Tree

```
Is the task language-based (text in → text out)?
├── YES → Start with an LLM API (Claude, GPT-4, Gemini)
│          ├── Works well? → Ship it
│          └── Too slow/costly/private? → Consider fine-tuning open model
└── NO  → Is it tabular/structured data?
           ├── YES → XGBoost / Random Forest / sklearn
           └── Is it images/audio/video?
                └── YES → Pretrained vision/audio model + fine-tune
```


## Summary

| Concept | Key takeaway |
|---|---|
| **Foundation models** | Large models pretrained on vast data; adaptable to many tasks |
| **LLM APIs** | Call via HTTP/Python SDK; no GPU required |
| **Prompt engineering** | System prompt + clear instructions + format spec + examples |
| **RAG** | Retrieve relevant docs → include in prompt → grounded answers |
| **LLM vs. train your own** | LLMs for language tasks; traditional ML for structured/tabular |